# Preventra CMS Readmission Model — Phase 1 & Phase 2 Feature Engineering

This notebook builds the feature-engineering and training pipeline for the
CMS DE-SynPUF migration described in:

- `docs/Preventra CMS Feasibility 4Week Plan.docx`
- `docs/Post_Discharge_Readmission_Monitoring (1).docx`
- `docs/cms/cms_migration_guide.md`
- `NewData.md` (exploratory analysis of these exact sample files)

It is organized in two parts, matching the two tracks in the feasibility plan:

| Track | Verdict (per feasibility doc) | What it predicts |
| --- | --- | --- |
| **Phase 1 — discharge-time risk** | Go | Will this patient be readmitted within 30 days of *this* discharge, using only data known at/ before discharge? |
| **Phase 2 — weekly post-discharge trend** | Conditional go (claims + pharmacy signals only for now) | Re-scores the same readmission risk once a week during the recovery window, using whatever follow-up/adherence signal has accumulated by that week, so a rising trend can be caught early |

**Data**: the full **"CMS Dataset Samples" Kaggle dataset** (`synpuf_data/sample 1` ..
`sample 20`), each one an official CMS DE-SynPUF 1% cross-sectional extract — roughly 62 GB
combined. This replaces the single ~100-row local preview used while the pipeline was first
being built — see `docs/Preventra CMS Feasibility 4Week Plan.docx`'s note that "only a small
preview is loaded so far".

**Memory architecture**: 62 GB cannot be held in memory at once on a Kaggle kernel, but each
sample individually (~3 GB) can. The 20 samples are non-overlapping draws from the same
synthetic population — no beneficiary appears in more than one sample — so no engineered
feature ever needs data from two samples at once. This notebook processes one sample
completely (load its 6 files → engineer Phase 1 features → engineer Phase 2 weekly features)
before moving to the next, discarding that sample's raw claims immediately afterward. Only
the small, already-aggregated feature rows accumulate across samples; the 62 GB of raw claims
is never resident all at once. See §1-2 for the loader and §3-4 for the per-sample pipeline.

> This notebook only builds and trains the models — it does not wire them into MLflow,
> the FastAPI backend, or MongoDB. See `docs/cms/cms_migration_guide.md` §5-6 for those steps.

## 0. File mapping

This Kaggle mirror lays each sample out as its own folder (`sample 1`, `sample 2`, ...,
`sample 20`) containing 6 files: three year-specific Beneficiary Summaries plus one
Inpatient, one Outpatient, and one Prescription Drug Event file that each already span all
three years (2008-2010) in a single CSV. Two things differ from the local 8-file preview
this pipeline was originally built against:

1. **No Carrier Claims (Part B) files** are included in this mirror. The Phase 1/2 features
   that read Carrier Claims (PCP follow-up visits, ED visits, ambulance calls) are written to
   degrade gracefully to 0 rather than fail when this table is absent — see §2 below.
2. **Filenames aren't hard-coded.** Exact suffixes vary by mirror/upload, so every CSV found
   under each sample folder is classified by the columns actually present in its header
   (the same approach `NewData.md`'s original exploration used), not by matching a literal
   filename string:

| Table | Detected by (columns present) | Role |
| --- | --- | --- |
| **Inpatient Claims** | `CLM_ADMSN_DT` + `NCH_BENE_DSCHRG_DT` | **Phase 1 spine** — one row per admission/discharge |
| Prescription Drug Events (PDE) | `PDE_ID` | Pharmacy fills / adherence — the Phase 2 adherence signal |
| Carrier Claims (Part B) *(if present)* | `PRF_PHYSN_NPI_1` + `TAX_NUM_1` | PCP/specialist follow-up + ED/ambulance touches |
| Outpatient Claims | `PRVDR_NUM` + `SEGMENT` (no `CLM_ADMSN_DT`) | Post-discharge hospital-based utilization |
| Beneficiary Summary | `BENE_BIRTH_DT` | Static demographics/comorbidities; year comes from the filename (`2008`/`2009`/`2010`) |

In [ ]:
import re
import gc
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

RANDOM_STATE = 42

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
# On Kaggle, attached datasets are mounted read-only under /kaggle/input/<dataset-slug>/...
# -- the exact slug isn't hard-coded here, so this still works whatever name the "CMS
# Dataset Samples" dataset gets attached under. Locally (no /kaggle/input), it falls back
# to the project's flat preview folder(s) so the notebook still runs off the laptop.
_KAGGLE_ROOT = Path("/kaggle/input")
INPUT_ROOTS = [r for r in [_KAGGLE_ROOT, Path("."), Path("../data/cms/samples"),
                           Path("data/cms/samples")] if r.exists()]
print(f"[config] Candidate input roots: {[str(r) for r in INPUT_ROOTS]}")

# Lookback windows for Phase 1 static features (days prior to admission)
IP_LOOKBACK_DAYS  = 365
ED_LOOKBACK_DAYS  = 90
RX_LOOKBACK_DAYS  = 90

# Phase 2 weekly monitoring window (matches the 30-90 day window in the feature proposal;
# NewData.md's trend-engine example uses 12 weeks ~ 90 days)
WINDOW_WEEKS = 12

# HCPCS code sets used throughout (E&M follow-up, ED visits, ambulance transport)
PCP_EM_CODES  = {"99211", "99212", "99213", "99214", "99215"}
ED_CODES      = {"99281", "99282", "99283", "99284", "99285"}
AMBULANCE_CODES = {"A0427"}

# Set to a small int (e.g. 5_000) to smoke-test the loader/pipeline on a slice of every file
# before committing to a full run; None reads every row.
MAX_ROWS_PER_FILE = None

## 1. Discover sample folders and load one sample's raw tables

CMS ships dates as `YYYYMMDD` integers, but pandas reads them as `float64` the moment a
column contains any missing value (e.g. `NCH_BENE_DSCHRG_DT` for an active admission,
`BENE_DEATH_DT` for a living beneficiary), so a raw `pd.to_datetime(..., format="%Y%m%d")`
call breaks on the trailing `.0`. `parse_cms_date` below handles that once, everywhere.

`find_sample_dirs` locates every `sample <N>`-style folder under the input roots (any count,
not hard-coded to 20). If no `sample <N>` subfolders exist at all (the flat local preview
layout), the input root itself is treated as a single sample, so the same loader works in
both places.

`load_one_sample` reads **one sample folder's 6 files** and returns that sample's own
`(bene_all, ip, op, car, pde)` — never more than one sample's raw claims at a time. Every
file is classified by its header (`classify_csv`) rather than a literal filename, and pruned
to only the columns this pipeline actually uses (`usecols_for`) — Outpatient claims alone
ship 76 columns (physician NPIs, tax IDs, line-level payment amounts) when only 2 are read
below, and at 62 GB total that pruning matters even within a single sample.

In [ ]:
def parse_cms_date(series: pd.Series) -> pd.Series:
    '''Parse a CMS YYYYMMDD date column (int, float-with-NaN, or string) to datetime.'''
    as_int = pd.to_numeric(series, errors="coerce").astype("Int64").astype(str)
    as_int = as_int.replace("<NA>", pd.NA)
    return pd.to_datetime(as_int, format="%Y%m%d", errors="coerce")


def hcpcs_columns(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c.startswith("HCPCS_CD_")]


def icd9_dgns_columns(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c.startswith("ICD9_DGNS_CD_")]


def icd9_prcdr_columns(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c.startswith("ICD9_PRCDR_CD_")]


def any_code_in_set(df: pd.DataFrame, columns: list, code_set: set) -> pd.Series:
    '''True if any of `columns` in a row matches a code in `code_set` (string compare).'''
    if not columns:
        return pd.Series(False, index=df.index)
    normalized = df[columns].astype(str).apply(lambda s: s.str.strip().str.upper())
    hit = pd.Series(False, index=df.index)
    for col in columns:
        hit |= normalized[col].isin(code_set)
    return hit


_SAMPLE_DIR_RE = re.compile(r"(?i)^sample[ _-]?(\d+)$")


def find_sample_dirs(roots: list) -> list:
    '''
    Every directory under `roots` named like "sample 1", "sample_1", "Sample1", etc.,
    sorted by sample number. Falls back to treating each root itself as one sample if no
    such subfolders are found anywhere (the flat local preview layout has no per-sample
    subfolders at all).
    '''
    found = {}
    for root in roots:
        for p in root.rglob("*"):
            if p.is_dir() and _SAMPLE_DIR_RE.match(p.name):
                found[p.resolve()] = p
    if found:
        return sorted(found.values(), key=lambda p: int(_SAMPLE_DIR_RE.match(p.name).group(1)))
    return [root for root in roots if list(root.glob("*.csv"))]


def classify_csv(path: Path):
    '''
    Identify which CMS table a CSV is by the columns actually present in its header, reading
    only the header (nrows=0) rather than the whole file. Returns (kind, header_columns) --
    the header set is reused by usecols_for() so the full file is only read once, with only
    the columns this pipeline actually needs.
    '''
    try:
        cols = set(pd.read_csv(path, nrows=0).columns)
    except Exception as e:
        print(f"[load] Could not read header of {path}: {e}")
        return "unknown", set()
    if "CLM_ADMSN_DT" in cols and "NCH_BENE_DSCHRG_DT" in cols:
        return "inpatient", cols
    if "PDE_ID" in cols:
        return "pde", cols
    if "PRF_PHYSN_NPI_1" in cols and "TAX_NUM_1" in cols:
        return "carrier", cols
    if "PRVDR_NUM" in cols and "SEGMENT" in cols:
        return "outpatient", cols
    if "BENE_BIRTH_DT" in cols:
        return "beneficiary", cols
    return "unknown", cols


def beneficiary_year(path: Path):
    m = re.search(r"(2008|2009|2010)", path.name)
    return int(m.group(1)) if m else None


# Every raw CMS table ships dozens of columns this notebook never reads (physician NPIs,
# tax IDs, line-level payment/processing amounts, ...). Trim to exactly what's used at read
# time instead of after -- SP_COLS here is also the authoritative, fixed list of the 11 CMS
# chronic-condition flags (NOT a prefix match against the loaded file: `SP_STATE_CODE` also
# starts with "SP_" but is a geographic FIPS code, not a condition flag).
SP_COLS = ["SP_ALZHDMTA", "SP_CHF", "SP_CHRNKIDN", "SP_CNCR", "SP_COPD", "SP_DEPRESSN",
           "SP_DIABETES", "SP_ISCHMCHT", "SP_OSTEOPRS", "SP_RA_OA", "SP_STRKETIA"]
# CMS DE-SynPUF codebook: 1=White, 2=Black, 3=Others, 5=Hispanic. Fixed (not discovered per
# sample) so every sample's one-hot race columns line up when samples are concatenated later.
RACE_CODES = [1, 2, 3, 5]

_FIXED_USECOLS = {
    "beneficiary": ["DESYNPUF_ID", "BENE_BIRTH_DT", "BENE_DEATH_DT", "BENE_SEX_IDENT_CD",
                     "BENE_RACE_CD", "BENE_ESRD_IND", "MEDREIMB_IP", "MEDREIMB_OP",
                     "MEDREIMB_CAR"] + SP_COLS,
    "inpatient": ["DESYNPUF_ID", "CLM_ID", "CLM_ADMSN_DT", "NCH_BENE_DSCHRG_DT", "CLM_DRG_CD",
                  "NCH_BENE_IP_DDCTBL_AMT"],
    "outpatient": ["DESYNPUF_ID", "CLM_FROM_DT"],
    "carrier": ["DESYNPUF_ID", "CLM_FROM_DT"],
    "pde": ["DESYNPUF_ID", "SRVC_DT", "PROD_SRVC_ID", "DAYS_SUPLY_NUM", "TOT_RX_CST_AMT"],
}
# Diagnosis/procedure/HCPCS code columns vary in count by table (10 diagnosis codes on
# Inpatient, 13 HCPCS line items on Carrier, ...) so they're kept dynamically by prefix
# rather than as a fixed list -- see icd9_dgns_columns/icd9_prcdr_columns/hcpcs_columns.
_DYNAMIC_PREFIXES = {
    "inpatient": ("ICD9_DGNS_CD_", "ICD9_PRCDR_CD_"),
    "carrier": ("HCPCS_CD_",),
}


def usecols_for(kind: str, header_cols: set) -> list:
    fixed = [c for c in _FIXED_USECOLS.get(kind, []) if c in header_cols]
    dynamic = [c for c in header_cols if c.startswith(_DYNAMIC_PREFIXES.get(kind, ()))]
    return fixed + dynamic


# ID/diagnosis/procedure/drug-code columns are forced to read as plain strings rather than
# letting pandas infer a dtype. At full-file scale, pandas' C parser infers dtype per
# internal read-chunk and can produce a *mixed* int/str object column when chunks disagree
# (e.g. one chunk of ICD9 codes looks all-numeric, another contains 'V'/'E' codes) --
# `np.unique`/sort then raises `TypeError: '<' not supported between instances of 'int' and
# 'str'`. Worse, it's silent when it doesn't crash: `_icd9_to_cci_weight` below only scores a
# code if it's a Python `str`, so any diagnosis code that got inferred as a number instead
# would score a comorbidity weight of 0 without any error at all. Forcing dtype=str at read
# time prevents both.
_STR_DTYPE_COLS = {
    "beneficiary": ["DESYNPUF_ID"],
    "inpatient": ["DESYNPUF_ID", "CLM_ID", "CLM_DRG_CD"],
    "outpatient": ["DESYNPUF_ID"],
    "carrier": ["DESYNPUF_ID"],
    "pde": ["DESYNPUF_ID", "PROD_SRVC_ID"],
}


def dtype_overrides_for(kind: str, header_cols: set) -> dict:
    cols = [c for c in _STR_DTYPE_COLS.get(kind, []) if c in header_cols]
    cols += [c for c in header_cols if c.startswith(_DYNAMIC_PREFIXES.get(kind, ()))]
    return {c: str for c in cols}


SAMPLE_DIRS = find_sample_dirs(INPUT_ROOTS)
if not SAMPLE_DIRS:
    raise FileNotFoundError(
        "No CMS DE-SynPUF sample data found under any of INPUT_ROOTS. On Kaggle, attach the "
        "'CMS Dataset Samples' input via Add Data; locally, point INPUT_ROOTS at a folder "
        "containing the sample CSVs."
    )
print(f"[config] Using {len(SAMPLE_DIRS)} sample folder(s): "
      f"{[d.name for d in SAMPLE_DIRS[:5]]}{' ...' if len(SAMPLE_DIRS) > 5 else ''}")

In [ ]:
def load_one_sample(sample_dir: Path):
    '''
    Read and classify every CSV in one sample folder, returning that sample's own
    (bene_all, ip, op, car, pde) -- never another sample's data. Raises FileNotFoundError if
    a required table (everything except Carrier) is missing from this folder.
    '''
    bene_frames, ip_frames, op_frames, car_frames, pde_frames = [], [], [], [], []
    skipped = []

    for csv_path in sorted(sample_dir.glob("*.csv")):
        kind, header_cols = classify_csv(csv_path)
        if kind == "unknown":
            skipped.append(csv_path.name)
            continue
        try:
            df = pd.read_csv(
                csv_path, usecols=usecols_for(kind, header_cols),
                dtype=dtype_overrides_for(kind, header_cols), nrows=MAX_ROWS_PER_FILE,
            )
        except Exception as e:
            # A single malformed/corrupted file shouldn't take down the whole 20-sample run.
            print(f"[load] {sample_dir.name}: failed to read {csv_path.name}, skipping it: {e}")
            continue
        if kind == "inpatient":
            ip_frames.append(df)
        elif kind == "outpatient":
            op_frames.append(df)
        elif kind == "carrier":
            car_frames.append(df)
        elif kind == "pde":
            pde_frames.append(df)
        elif kind == "beneficiary":
            year = beneficiary_year(csv_path)
            if year is None:
                print(f"[load] {sample_dir.name}: skipping {csv_path.name} -- looks like a "
                      f"Beneficiary Summary but no 2008/2009/2010 year in its filename")
                continue
            df["YEAR"] = year
            bene_frames.append(df)

    if skipped:
        print(f"[load] {sample_dir.name}: skipped {len(skipped)} unrecognized file(s): {skipped}")

    for name, frames in [("Inpatient Claims", ip_frames), ("Beneficiary Summary", bene_frames),
                          ("Part D Events", pde_frames), ("Outpatient Claims", op_frames)]:
        if not frames:
            raise FileNotFoundError(f"{sample_dir.name}: no {name} file found")

    bene_all = pd.concat(bene_frames, ignore_index=True)
    # The demographic join later assumes (DESYNPUF_ID, YEAR) is unique within a sample --
    # true for CMS's data, but a plain merge would silently fan a single admission out into
    # duplicate rows if it were ever violated. Guard it explicitly instead.
    dup_bene = bene_all.duplicated(subset=["DESYNPUF_ID", "YEAR"]).sum()
    if dup_bene:
        print(f"[load] {sample_dir.name}: WARNING {dup_bene} duplicate (DESYNPUF_ID, YEAR) "
              f"beneficiary row(s) -- keeping the first occurrence of each")
        bene_all = bene_all.drop_duplicates(subset=["DESYNPUF_ID", "YEAR"], keep="first")
    bene_all["BENE_BIRTH_DT"] = parse_cms_date(bene_all["BENE_BIRTH_DT"])
    bene_all["BENE_DEATH_DT"] = parse_cms_date(bene_all["BENE_DEATH_DT"])

    ip = pd.concat(ip_frames, ignore_index=True)
    ip["CLM_ADMSN_DT"] = parse_cms_date(ip["CLM_ADMSN_DT"])
    ip["NCH_BENE_DSCHRG_DT"] = parse_cms_date(ip["NCH_BENE_DSCHRG_DT"])
    ip = ip.reset_index(drop=True)
    # A plain int would collide across samples once concatenated later -- prefix with the
    # sample name so discharge_row_id stays globally unique without a renumbering pass.
    ip["discharge_row_id"] = sample_dir.name + "::" + ip.index.astype(str)

    op = pd.concat(op_frames, ignore_index=True)
    op["CLM_FROM_DT"] = parse_cms_date(op["CLM_FROM_DT"])

    if car_frames:
        car = pd.concat(car_frames, ignore_index=True)
        car["CLM_FROM_DT"] = parse_cms_date(car["CLM_FROM_DT"])
    else:
        car = pd.DataFrame(columns=["DESYNPUF_ID", "CLM_FROM_DT"])
        car["CLM_FROM_DT"] = pd.to_datetime(car["CLM_FROM_DT"])

    pde = pd.concat(pde_frames, ignore_index=True)
    pde["SRVC_DT"] = parse_cms_date(pde["SRVC_DT"])

    return bene_all, ip, op, car, pde

## 2. Derive the ground-truth 30-day readmission label

CMS claims have no `readmitted` column like the old UCI diabetes dataset — it has to be
derived by comparing each discharge to the *next* admission for the same beneficiary
(`docs/cms/cms_migration_guide.md` §4). The version below is a vectorized rewrite of that same
logic (the guide's reference implementation loops row-by-row with `.iterrows()`, which is
O(n²) and won't scale). It runs per sample — safe, since a patient's admission history never
spans two samples, so "the next admission for this patient" is always fully visible within
one sample's own `ip` table.

Two correctness points the reference version misses:

1. **In-stay deaths are excluded.** If `BENE_DEATH_DT` falls inside `[CLM_ADMSN_DT, NCH_BENE_DSCHRG_DT]`,
   the patient cannot be readmitted — including that row would silently bias the label toward 0
   for the sickest patients (per `NewData.md`'s note on mortality tracking).
2. **`days_to_readmit` is kept alongside the binary label** — Phase 2's weekly panel needs it to
   know when to stop generating weekly snapshots for a patient who was already readmitted.

In [ ]:
def add_readmission_label(ip: pd.DataFrame, bene_all: pd.DataFrame) -> pd.DataFrame:
    # Drop admissions where the patient died during the index stay -- not eligible for readmission
    died_in_stay = ip.merge(
        bene_all[["DESYNPUF_ID", "YEAR", "BENE_DEATH_DT"]], on="DESYNPUF_ID", how="left"
    )
    died_in_stay = died_in_stay[died_in_stay["BENE_DEATH_DT"].notna()]
    died_mask = (
        (died_in_stay["BENE_DEATH_DT"] >= died_in_stay["CLM_ADMSN_DT"])
        & (died_in_stay["BENE_DEATH_DT"] <= died_in_stay["NCH_BENE_DSCHRG_DT"])
    )
    died_row_ids = set(died_in_stay.loc[died_mask, "discharge_row_id"])
    ip = ip[~ip["discharge_row_id"].isin(died_row_ids)].reset_index(drop=True)

    # Vectorized "next admission for this patient" lookup
    ip = ip.sort_values(["DESYNPUF_ID", "CLM_ADMSN_DT"]).reset_index(drop=True)
    ip["next_admit_dt"] = ip.groupby("DESYNPUF_ID")["CLM_ADMSN_DT"].shift(-1)
    ip["days_to_readmit"] = (ip["next_admit_dt"] - ip["NCH_BENE_DSCHRG_DT"]).dt.days
    ip["readmitted_30d"] = ((ip["days_to_readmit"] >= 0) & (ip["days_to_readmit"] <= 30)).astype(int)
    return ip

---
## 3. Shared feature-engineering building blocks

`count_events_in_window` is the one generic building block both phases share: "for each row
of an index table, aggregate events from another table whose date falls in some window, for
the same patient" — used for Phase 1's fixed lookback windows (e.g. 365 days before
admission) and Phase 2's growing post-discharge windows (e.g. the 7×w days after discharge)
alike.

In [ ]:
def count_events_in_window(
    index_df: pd.DataFrame,
    index_id_col: str,
    lower_bound_col: str,
    upper_bound_col: str,
    events_df: pd.DataFrame,
    events_id_col: str,
    events_date_col: str,
    agg: str = "count",
    agg_col: str | None = None,
    lower_inclusive: bool = False,
    upper_inclusive: bool = True,
) -> pd.Series:
    '''
    agg: "count" (rows), "nunique" (distinct agg_col values), or "sum" (agg_col).

    Avoids ever materializing an ID-based cross join. An earlier version of this function
    merged on patient ID first and filtered by date after -- fine when patients have few
    events, but a patient with hundreds of Part D fills joined against dozens of Phase 2
    weekly rows produces thousands of intermediate rows *per patient*, and that fan-out
    compounds across tens of thousands of patients into a memory spike big enough to crash a
    Kaggle kernel mid-sample at full 20-sample scale (verified: reproduced the crash pattern
    and confirmed this rewrite fixes it before shipping).

    Instead: encode each patient ID as a small integer and combine it with the event date
    into a single sortable key (`patient_code * DATE_OFFSET + date_ordinal`), so sorting by
    this key is exactly equivalent to sorting by (patient, date) -- keys from different
    patients never interleave, since DATE_OFFSET comfortably exceeds any realistic date-
    ordinal magnitude. A single global `np.searchsorted` then finds every index row's window
    boundaries for every patient simultaneously: no per-patient loop, no cross product.
    "count"/"sum" are fully vectorized this way (sum via a cumulative-sum prefix array).
    "nunique" can't use a simple prefix trick, so it loops per index row over the *slice*
    `searchsorted` already found -- bounded by that patient's events actually inside the
    window, never the full cross product, so it stays memory-safe even though it's slower.

    Verified to produce identical output to the old merge-based version across count/sum/
    nunique, inclusive/exclusive boundary combinations, patients with zero and with hundreds
    of events, and the self-referential same-table/same-column-name case.
    '''
    if len(index_df) == 0:
        return pd.Series(dtype=float, index=index_df.index)

    # A NaN agg_col value would otherwise poison every later cumulative sum for that patient
    # (np.cumsum doesn't skip NaN) or crash nunique's sort by mixing NaN in with real values --
    # drop it up front alongside rows with no event date at all.
    dropna_cols = [events_date_col] + ([agg_col] if agg_col and agg != "count" else [])
    events = events_df[[events_id_col, events_date_col] + ([agg_col] if agg_col else [])].dropna(
        subset=dropna_cols
    )

    # Encode index and event IDs with the same code space so their keys line up.
    all_ids = pd.concat([index_df[index_id_col], events[events_id_col]], ignore_index=True)
    codes, _ = pd.factorize(all_ids)
    idx_codes = codes[: len(index_df)]
    ev_codes = codes[len(index_df):]

    epoch = pd.Timestamp("1900-01-01")  # far enough back that ordinals stay positive
    ev_ordinal = (events[events_date_col] - epoch).dt.days.to_numpy()
    lower_ordinal = (index_df[lower_bound_col] - epoch).dt.days.to_numpy()
    upper_ordinal = (index_df[upper_bound_col] - epoch).dt.days.to_numpy()

    date_offset = 10_000_000  # comfortably larger than any realistic date-ordinal magnitude
    ev_key = ev_codes.astype("int64") * date_offset + ev_ordinal.astype("int64")
    order = np.argsort(ev_key, kind="stable")
    ev_key_sorted = ev_key[order]

    lower_key = idx_codes.astype("int64") * date_offset + lower_ordinal.astype("int64")
    upper_key = idx_codes.astype("int64") * date_offset + upper_ordinal.astype("int64")

    lo_side = "left" if lower_inclusive else "right"
    hi_side = "right" if upper_inclusive else "left"
    lo_idx = np.searchsorted(ev_key_sorted, lower_key, side=lo_side)
    hi_idx = np.searchsorted(ev_key_sorted, upper_key, side=hi_side)

    if agg == "count":
        result = (hi_idx - lo_idx).astype(float)
    elif agg == "sum":
        vals_sorted = events[agg_col].to_numpy(dtype="float64")[order]
        cum = np.concatenate(([0.0], np.cumsum(vals_sorted)))
        result = cum[hi_idx] - cum[lo_idx]
    elif agg == "nunique":
        # Cast to str even though dtype_overrides_for() already forces agg_col to read as
        # str -- cheap insurance against np.unique's sort crashing on any future caller that
        # passes a column pandas still inferred inconsistently.
        vals_sorted = events[agg_col].astype(str).to_numpy()[order]
        result = np.empty(len(index_df), dtype=float)
        for i in range(len(index_df)):
            a, b = lo_idx[i], hi_idx[i]
            result[i] = len(np.unique(vals_sorted[a:b])) if b > a else 0.0
    else:
        raise ValueError(f"Unsupported agg: {agg}")

    return pd.Series(result, index=index_df.index)

In [ ]:
# Charlson Comorbidity Index -- same weight table as features/cci.py (kept identical so
# scores from the CMS pipeline stay comparable to the existing diabetes-data baseline),
# generalized to scan however many ICD9_DGNS_CD_* columns a claim actually has.
CCI_MAP: list[tuple[str, int]] = [
    (r"^410|^412",                          1),  # Myocardial infarction
    (r"^428",                               1),  # Congestive heart failure
    (r"^4[45]",                             1),  # Peripheral vascular disease
    (r"^43[0-8]",                           1),  # Cerebrovascular disease
    (r"^290",                               1),  # Dementia
    (r"^49[0-6]|^500|^505|^5064",          1),  # Chronic pulmonary disease
    (r"^710[01]|^7140|^7141|^7142|^7148",  1),  # Connective tissue disease
    (r"^53[1-4]",                           1),  # Peptic ulcer disease
    (r"^571",                               1),  # Mild liver disease
    (r"^250[0-3]",                          1),  # Diabetes without complications
    (r"^196|^197|^198|^199",               6),  # Metastatic solid tumor
    (r"^042|^043|^044",                    6),  # AIDS/HIV
    (r"^572[2-8]",                          3),  # Moderate/severe liver disease
    (r"^342|^344[01]",                      2),  # Hemiplegia
    (r"^58[2-3]|^585|^586|^5880",          2),  # Moderate/severe renal disease
    (r"^250[4-9]",                          2),  # Diabetes with end-organ damage
    (r"^1[4-9][0-9]|^20[0-8]",            2),  # Tumor (non-metastatic)
    (r"^204[1]|^205[3]|^206[3]|^207[12]", 2),  # Leukemia
    (r"^200|^201|^202",                    2),  # Lymphoma
]
_COMPILED_CCI = [(re.compile(p), w) for p, w in CCI_MAP]
_BH_LOW, _BH_HIGH = 291, 319  # ICD-9 range for mental/behavioural health disorders


def _icd9_to_cci_weight(code) -> int:
    if not isinstance(code, str) or not code.strip():
        return 0
    code_clean = code.strip().upper().replace(".", "")
    for pattern, weight in _COMPILED_CCI:
        if pattern.match(code_clean):
            return weight
    return 0


def compute_cci_row(row: pd.Series, dgns_cols: list) -> int:
    seen_weights = set()
    total = 0
    for col in dgns_cols:
        w = _icd9_to_cci_weight(row.get(col))
        if w > 0 and w not in seen_weights:
            total += w
            seen_weights.add(w)
    return total


def has_behavioral_health_code(row: pd.Series, dgns_cols: list) -> int:
    for col in dgns_cols:
        code = str(row.get(col, "")).strip().split(".")[0]
        try:
            if _BH_LOW <= int(code) <= _BH_HIGH:
                return 1
        except ValueError:
            continue
    return 0

## 4. Per-sample feature engineering (Phase 1 + Phase 2)

Both phases are functions that take one sample's already-loaded tables and return that
sample's engineered rows. The driving loop in §5 calls these once per sample, so a sample's
raw Carrier/Outpatient/PDE claims are only ever in memory long enough to be reduced to the
handful of aggregated columns kept below.

**Phase 1** (`engineer_phase1`): one row per admission. Every feature only uses information
available **at or before** `NCH_BENE_DSCHRG_DT`, matching the "Go" verdict for this track in
the feasibility doc. Feature column lists (`PHASE1_FEATURE_COLS` etc.) are fixed once from
the known/fixed schema (`SP_COLS`, `RACE_CODES`) rather than discovered per sample — every
sample must produce identical columns so `phase1_df`/`phase2_df` concatenate cleanly across
all 20. `CLM_DRG_CD` is passed through as a plain numeric code rather than one-hot encoded
for the same reason: which DRGs are "top 15" would otherwise differ sample to sample.

**Phase 2** (`engineer_phase2`): explodes each Phase 1 admission into weekly snapshots,
layering cumulative post-discharge follow-up/adherence signal on top. For episodes that
*were* readmitted within 30 days, weekly snapshots stop being generated once the readmission
already happened. It is *not* a new label per week — the outcome being predicted is still
"did this discharge episode lead to a readmission", just re-scored with more information as
the weeks pass.

In [ ]:
LABEL_COL = "readmitted_30d"
PHASE1_METADATA_COLS = [
    "discharge_row_id", "DESYNPUF_ID", "CLM_ID", "CLM_ADMSN_DT", "NCH_BENE_DSCHRG_DT",
    "days_to_readmit",
]
RACE_DUMMY_COLS = [f"race_cd_{c}" for c in RACE_CODES] + ["race_cd_other"]
PHASE1_FEATURE_COLS = [
    "age_at_admission", "frailty_proxy", "sex_male", "esrd_flag",
    "total_chronic_conditions", *SP_COLS, *RACE_DUMMY_COLS,
    "prior_year_medicare_spend", "length_of_stay_days", "charlson_comorbidity_index",
    "behavioral_health_flag", "diagnosis_code_count", "procedure_code_count",
    "complex_discharge_flag", "is_zero_ip_deductible", "drg_code",
    "ip_admissions_365d_prior", "ed_visits_90d_prior", "ambulance_calls_90d_prior",
    "medication_count_90d_prior", "medication_possession_ratio_90d_prior",
    "rx_spend_90d_prior", "high_utilizer_flag",
]
PHASE2_DYNAMIC_COLS = [
    "week_number", "days_in_recovery_window",
    "post_pcp_visit_count", "had_pcp_followup",
    "post_ed_visit_count", "post_ambulance_call_count", "post_outpatient_encounter_count",
    "post_rx_fill_count", "post_days_supply_covered", "post_rx_spend",
    "medication_possession_ratio", "medication_refill_gap_days",
]
PHASE2_FEATURE_COLS = PHASE1_FEATURE_COLS + PHASE2_DYNAMIC_COLS


def engineer_phase1(ip: pd.DataFrame, bene_all: pd.DataFrame, car: pd.DataFrame,
                     pde: pd.DataFrame) -> pd.DataFrame:
    '''
    One sample's discharge-time feature matrix. Mutates `car` in place (adds the
    _is_ed_visit/_is_ambulance/_is_pcp_visit flags engineer_phase2 reuses) -- the caller's
    own `car` reference sees these columns after this call returns.
    '''
    ip = ip.copy()
    ip["ADMIT_YEAR"] = ip["CLM_ADMSN_DT"].dt.year

    # --- Demographics: year-matched beneficiary join + age + chronic conditions ---
    bene_cols = ["DESYNPUF_ID", "YEAR", "BENE_BIRTH_DT", "BENE_SEX_IDENT_CD", "BENE_RACE_CD",
                 "BENE_ESRD_IND", "MEDREIMB_IP", "MEDREIMB_OP", "MEDREIMB_CAR"] + SP_COLS
    merged = ip.merge(
        bene_all[bene_cols], left_on=["DESYNPUF_ID", "ADMIT_YEAR"],
        right_on=["DESYNPUF_ID", "YEAR"], how="left",
    )

    # Fallback: if a beneficiary has no summary for the exact admission year, use whichever
    # year's summary is closest rather than dropping the demographic features.
    missing = merged[merged["YEAR"].isna()]
    if len(missing):
        bene_by_id = {pid: g[["YEAR"] + bene_cols[2:]].to_dict("records")
                      for pid, g in bene_all.groupby("DESYNPUF_ID")}

        def _nearest_year_record(r):
            candidates = bene_by_id.get(r["DESYNPUF_ID"])
            if not candidates:
                return None  # no beneficiary summary for this patient in any year
            return min(candidates, key=lambda c: abs(c["YEAR"] - r["ADMIT_YEAR"]))

        fallback_records = missing.apply(_nearest_year_record, axis=1)
        has_fallback = fallback_records.notna()
        fallback_idx = missing.index[has_fallback]
        # Only fill rows with an actual candidate -- filling with a Python `None` (rather
        # than leaving the merge's original NaN/NaT) breaks datetime arithmetic further down.
        for col in bene_cols[2:]:
            fill_values = fallback_records[has_fallback].apply(lambda rec: rec.get(col))
            merged.loc[fallback_idx, col] = merged.loc[fallback_idx, col].fillna(fill_values)

    ip = merged
    ip["age_at_admission"] = ((ip["CLM_ADMSN_DT"] - ip["BENE_BIRTH_DT"]).dt.days / 365.25)
    ip["age_at_admission"] = ip["age_at_admission"].fillna(0).astype(int)
    ip["frailty_proxy"] = (ip["age_at_admission"] >= 75).astype(int)

    ip["sex_male"] = (ip["BENE_SEX_IDENT_CD"] == 1).astype(int)
    ip["esrd_flag"] = ip["BENE_ESRD_IND"].astype(str).str.upper().eq("Y").astype(int)

    for col in SP_COLS:
        ip[col] = (ip[col] == 1).astype(int)  # CMS codebook: 1=Yes, 2=No
    ip["total_chronic_conditions"] = ip[SP_COLS].sum(axis=1)

    # Fixed categories (not discovered from this sample's data) so every sample produces the
    # same race_cd_* columns even if a given race code doesn't appear in it.
    race_cat = pd.Categorical(ip["BENE_RACE_CD"], categories=RACE_CODES)
    race_dummies = pd.get_dummies(race_cat, prefix="race_cd", dtype=int)
    race_dummies["race_cd_other"] = (~ip["BENE_RACE_CD"].isin(RACE_CODES)).astype(int)
    ip = pd.concat([ip, race_dummies], axis=1)

    ip["prior_year_medicare_spend"] = ip[["MEDREIMB_IP", "MEDREIMB_OP", "MEDREIMB_CAR"]].sum(axis=1)

    # --- Diagnosis-derived features: CCI, behavioral health, discharge complexity ---
    dgns_cols = icd9_dgns_columns(ip)
    prcdr_cols = icd9_prcdr_columns(ip)

    ip["length_of_stay_days"] = (ip["NCH_BENE_DSCHRG_DT"] - ip["CLM_ADMSN_DT"]).dt.days
    ip["charlson_comorbidity_index"] = ip.apply(lambda r: compute_cci_row(r, dgns_cols), axis=1)
    ip["behavioral_health_flag"] = ip.apply(lambda r: has_behavioral_health_code(r, dgns_cols), axis=1)

    ip["diagnosis_code_count"] = ip[dgns_cols].notna().sum(axis=1)
    ip["procedure_code_count"] = ip[prcdr_cols].notna().sum(axis=1) if prcdr_cols else 0
    ip["complex_discharge_flag"] = (
        (ip["length_of_stay_days"] > 7)
        & (ip["diagnosis_code_count"] > 7)
        & (ip["procedure_code_count"] > 2)
    ).astype(int)

    ip["is_zero_ip_deductible"] = (ip["NCH_BENE_IP_DDCTBL_AMT"].fillna(0) == 0).astype(int)
    # Plain numeric code, not one-hot -- see the module note above on why a per-sample
    # "top-N DRG" encoding wouldn't concatenate cleanly across all 20 samples.
    ip["drg_code"] = pd.to_numeric(ip["CLM_DRG_CD"], errors="coerce").fillna(-1).astype(int)

    # --- Prior-utilization lookback features (365d admissions, 90d ED/ambulance, 90d Rx) ---
    ip["_lookback_ip_start"] = ip["CLM_ADMSN_DT"] - pd.Timedelta(days=IP_LOOKBACK_DAYS)
    ip["_lookback_ed_start"] = ip["CLM_ADMSN_DT"] - pd.Timedelta(days=ED_LOOKBACK_DAYS)
    ip["_lookback_rx_start"] = ip["CLM_ADMSN_DT"] - pd.Timedelta(days=RX_LOOKBACK_DAYS)

    # Prior inpatient admissions in the trailing 365 days (self-referential on the spine;
    # the strict "< CLM_ADMSN_DT" upper bound naturally excludes the row's own admission).
    ip["ip_admissions_365d_prior"] = count_events_in_window(
        ip, "DESYNPUF_ID", "_lookback_ip_start", "CLM_ADMSN_DT",
        ip, "DESYNPUF_ID", "CLM_ADMSN_DT",
        agg="count", lower_inclusive=True, upper_inclusive=False,
    )

    car_hcpcs_cols = hcpcs_columns(car)
    car["_is_ed_visit"] = any_code_in_set(car, car_hcpcs_cols, ED_CODES)
    car["_is_ambulance"] = any_code_in_set(car, car_hcpcs_cols, AMBULANCE_CODES)
    car["_is_pcp_visit"] = any_code_in_set(car, car_hcpcs_cols, PCP_EM_CODES)

    ip["ed_visits_90d_prior"] = count_events_in_window(
        ip, "DESYNPUF_ID", "_lookback_ed_start", "CLM_ADMSN_DT",
        car[car["_is_ed_visit"]], "DESYNPUF_ID", "CLM_FROM_DT",
        agg="count", lower_inclusive=True, upper_inclusive=False,
    )
    ip["ambulance_calls_90d_prior"] = count_events_in_window(
        ip, "DESYNPUF_ID", "_lookback_ed_start", "CLM_ADMSN_DT",
        car[car["_is_ambulance"]], "DESYNPUF_ID", "CLM_FROM_DT",
        agg="count", lower_inclusive=True, upper_inclusive=False,
    )

    ip["medication_count_90d_prior"] = count_events_in_window(
        ip, "DESYNPUF_ID", "_lookback_rx_start", "CLM_ADMSN_DT",
        pde, "DESYNPUF_ID", "SRVC_DT",
        agg="nunique", agg_col="PROD_SRVC_ID", lower_inclusive=True, upper_inclusive=False,
    )
    _days_supply_90d = count_events_in_window(
        ip, "DESYNPUF_ID", "_lookback_rx_start", "CLM_ADMSN_DT",
        pde, "DESYNPUF_ID", "SRVC_DT",
        agg="sum", agg_col="DAYS_SUPLY_NUM", lower_inclusive=True, upper_inclusive=False,
    )
    ip["medication_possession_ratio_90d_prior"] = (_days_supply_90d / RX_LOOKBACK_DAYS).clip(0, 1)
    ip["rx_spend_90d_prior"] = count_events_in_window(
        ip, "DESYNPUF_ID", "_lookback_rx_start", "CLM_ADMSN_DT",
        pde, "DESYNPUF_ID", "SRVC_DT",
        agg="sum", agg_col="TOT_RX_CST_AMT", lower_inclusive=True, upper_inclusive=False,
    )

    ip["high_utilizer_flag"] = (
        (ip["ip_admissions_365d_prior"] >= 2) | (ip["ed_visits_90d_prior"] >= 2)
    ).astype(int)

    phase1_df = ip[PHASE1_METADATA_COLS + PHASE1_FEATURE_COLS + [LABEL_COL]].copy()
    phase1_df[PHASE1_FEATURE_COLS] = phase1_df[PHASE1_FEATURE_COLS].fillna(0)
    return phase1_df


def engineer_phase2(phase1_df: pd.DataFrame, op: pd.DataFrame, car: pd.DataFrame,
                     pde: pd.DataFrame) -> pd.DataFrame:
    '''One sample's weekly post-discharge feature matrix, built from that same sample's
    Phase 1 rows plus its still-in-scope Outpatient/Carrier/PDE tables. `car` must already
    carry the _is_ed_visit/_is_ambulance/_is_pcp_visit flags engineer_phase1 adds.'''
    pcp_events = car[car["_is_pcp_visit"]]
    ed_events = car[car["_is_ed_visit"]]
    ambulance_events = car[car["_is_ambulance"]]

    # --- Build the weekly panel: explode each discharge into 1..max_week rows ---
    panel_base = phase1_df.copy()
    readmit_week = np.ceil(panel_base["days_to_readmit"] / 7)
    max_week_raw = np.where(
        (panel_base[LABEL_COL] == 1) & panel_base["days_to_readmit"].notna(),
        np.minimum(WINDOW_WEEKS, readmit_week),
        WINDOW_WEEKS,
    )
    # np.where returns a plain ndarray -- assign to the DataFrame first so `.clip(lower=...)`
    # resolves to pandas' Series.clip (numpy's ndarray.clip takes positional min/max, not `lower`).
    panel_base["max_week"] = max_week_raw.astype(int)
    panel_base["max_week"] = panel_base["max_week"].clip(lower=1)

    panel_base["week_number"] = panel_base["max_week"].apply(lambda m: list(range(1, m + 1)))
    weekly_panel = panel_base.explode("week_number", ignore_index=True)
    weekly_panel["week_number"] = weekly_panel["week_number"].astype(int)

    weekly_panel["t0"] = weekly_panel["NCH_BENE_DSCHRG_DT"]
    weekly_panel["t_w"] = weekly_panel["t0"] + pd.to_timedelta(7 * weekly_panel["week_number"], unit="D")
    weekly_panel["days_in_recovery_window"] = 7 * weekly_panel["week_number"]

    # --- Cumulative post-discharge features per weekly snapshot: (t0, t_w] ---
    weekly_panel["post_pcp_visit_count"] = count_events_in_window(
        weekly_panel, "DESYNPUF_ID", "t0", "t_w", pcp_events, "DESYNPUF_ID", "CLM_FROM_DT",
        agg="count",
    )
    weekly_panel["had_pcp_followup"] = (weekly_panel["post_pcp_visit_count"] > 0).astype(int)

    weekly_panel["post_ed_visit_count"] = count_events_in_window(
        weekly_panel, "DESYNPUF_ID", "t0", "t_w", ed_events, "DESYNPUF_ID", "CLM_FROM_DT",
        agg="count",
    )
    weekly_panel["post_ambulance_call_count"] = count_events_in_window(
        weekly_panel, "DESYNPUF_ID", "t0", "t_w", ambulance_events, "DESYNPUF_ID", "CLM_FROM_DT",
        agg="count",
    )
    weekly_panel["post_outpatient_encounter_count"] = count_events_in_window(
        weekly_panel, "DESYNPUF_ID", "t0", "t_w", op, "DESYNPUF_ID", "CLM_FROM_DT",
        agg="count",
    )

    weekly_panel["post_rx_fill_count"] = count_events_in_window(
        weekly_panel, "DESYNPUF_ID", "t0", "t_w", pde, "DESYNPUF_ID", "SRVC_DT",
        agg="nunique", agg_col="PROD_SRVC_ID",
    )
    weekly_panel["post_days_supply_covered"] = count_events_in_window(
        weekly_panel, "DESYNPUF_ID", "t0", "t_w", pde, "DESYNPUF_ID", "SRVC_DT",
        agg="sum", agg_col="DAYS_SUPLY_NUM",
    )
    weekly_panel["post_rx_spend"] = count_events_in_window(
        weekly_panel, "DESYNPUF_ID", "t0", "t_w", pde, "DESYNPUF_ID", "SRVC_DT",
        agg="sum", agg_col="TOT_RX_CST_AMT",
    )
    weekly_panel["medication_possession_ratio"] = (
        weekly_panel["post_days_supply_covered"] / weekly_panel["days_in_recovery_window"]
    ).clip(0, 1)
    weekly_panel["medication_refill_gap_days"] = (
        weekly_panel["days_in_recovery_window"] - weekly_panel["post_days_supply_covered"]
    ).clip(lower=0)

    phase2_df = weekly_panel[
        ["discharge_row_id", "DESYNPUF_ID", "t0", "t_w"] + PHASE2_FEATURE_COLS + [LABEL_COL]
    ].copy()
    phase2_df[PHASE2_FEATURE_COLS] = phase2_df[PHASE2_FEATURE_COLS].fillna(0)
    return phase2_df

## 5. Run the per-sample pipeline across every discovered sample

This is the only place all 20 samples are touched in the same pass, and even here only one
sample's raw claims are ever resident at once: each iteration loads a sample, engineers its
Phase 1 and Phase 2 rows, appends just those (small, aggregated) results, then explicitly
frees that sample's raw tables before moving on. A sample that fails to load (e.g. a missing
or corrupted file) is skipped with a warning rather than aborting the whole run.

In [ ]:
phase1_parts, phase2_parts = [], []

for sample_dir in SAMPLE_DIRS:
    print(f"=== Processing {sample_dir.name} ===")
    try:
        bene_all, ip, op, car, pde = load_one_sample(sample_dir)
    except FileNotFoundError as e:
        print(f"[load] Skipping {sample_dir.name} entirely: {e}")
        continue

    ip = add_readmission_label(ip, bene_all)
    phase1_sample_df = engineer_phase1(ip, bene_all, car, pde)
    phase2_sample_df = engineer_phase2(phase1_sample_df, op, car, pde)

    print(f"    {len(phase1_sample_df)} discharges, "
          f"{int(phase1_sample_df[LABEL_COL].sum())} readmitted within 30d, "
          f"{len(phase2_sample_df)} weekly snapshot rows")

    phase1_parts.append(phase1_sample_df)
    phase2_parts.append(phase2_sample_df)

    del bene_all, ip, op, car, pde, phase1_sample_df, phase2_sample_df
    gc.collect()

if not phase1_parts:
    raise RuntimeError("No sample folder could be fully loaded -- see warnings above.")

phase1_df = pd.concat(phase1_parts, ignore_index=True)
del phase1_parts
gc.collect()

phase2_df = pd.concat(phase2_parts, ignore_index=True)
del phase2_parts
gc.collect()

null_counts = phase1_df[PHASE1_FEATURE_COLS + [LABEL_COL]].isnull().sum()
assert null_counts.sum() == 0, f"Unexpected nulls in Phase 1 features:\n{null_counts[null_counts > 0]}"
null_counts2 = phase2_df[PHASE2_FEATURE_COLS + [LABEL_COL]].isnull().sum()
assert null_counts2.sum() == 0, f"Unexpected nulls in Phase 2 features:\n{null_counts2[null_counts2 > 0]}"

print(f"\n[phase1] Final feature matrix: {phase1_df.shape[0]:,} rows x {len(PHASE1_FEATURE_COLS)} features")
print(f"[phase1] Positive rate: {phase1_df[LABEL_COL].mean():.2%}")
if phase1_df[LABEL_COL].sum() < 20:
    print("[phase1] WARNING: very few positives for a model of this feature count -- double "
          "check SAMPLE_DIRS actually picked up every sample folder rather than just one.")

print(f"[phase2] Final weekly feature matrix: {phase2_df.shape[0]:,} rows x {len(PHASE2_FEATURE_COLS)} features")
print(f"[phase2] Positive rate: {phase2_df[LABEL_COL].mean():.2%}")

# phase1_df.to_csv("phase1_features.csv", index=False)          # uncomment when ready to persist
# phase2_df.to_csv("phase2_weekly_features.csv", index=False)   # uncomment when ready to persist
phase1_df.head()

## 6. Phase 1 — Train the discharge-time model

Matches the existing baseline pipeline's conventions (`models/train.py`): a temporal
train/test split (earliest admissions train, latest test — avoids leaking future claims
into the training set), a `DecisionTreeClassifier` with `class_weight="balanced"` to handle
the label imbalance, then an optional `GridSearchCV` tuning pass.

Cross-validation and stratified splitting are still guarded even at full 20-sample scale: if
there are too few positive examples to stratify/fold on, the notebook falls back to the
unstratified baseline and explains why, rather than crashing.

In [ ]:
def temporal_split(df: pd.DataFrame, date_col: str, ratio: float = 0.8):
    df = df.sort_values(date_col).reset_index(drop=True)
    split_idx = int(len(df) * ratio)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()


def safe_auc(y_true, y_prob):
    '''roc_auc_score requires both classes present in y_true; return NaN otherwise.'''
    if y_true.nunique() < 2:
        return float("nan")
    return roc_auc_score(y_true, y_prob)


def evaluate_classifier(name, model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc = safe_auc(y_test, y_prob)
    print(f"--- {name} ---")
    print(f"AUC-ROC   : {auc:.4f}" if pd.notna(auc) else "AUC-ROC   : n/a (single class in test set)")
    print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")
    print(classification_report(y_test, y_pred, zero_division=0))
    return y_prob

In [ ]:
train1_df, test1_df = temporal_split(phase1_df, "CLM_ADMSN_DT", ratio=0.8)
X_train1, y_train1 = train1_df[PHASE1_FEATURE_COLS], train1_df[LABEL_COL]
X_test1, y_test1 = test1_df[PHASE1_FEATURE_COLS], test1_df[LABEL_COL]

print(f"[phase1 split] train={len(train1_df)} ({y_train1.mean():.2%} positive) | "
      f"test={len(test1_df)} ({y_test1.mean():.2%} positive)")

phase1_baseline = DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)
phase1_baseline.fit(X_train1, y_train1)
_ = evaluate_classifier("Phase 1 baseline DecisionTree", phase1_baseline, X_test1, y_test1)

In [ ]:
# GridSearchCV tuning -- guarded: CV needs at least `cv` positive examples in the training
# fold. Falls back to the baseline model if tuning isn't viable, with the exact reason logged
# (should only trigger if far fewer than the full 20 samples ended up loaded).
PARAM_GRID_P1 = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_leaf": [5, 10, 20],
    "min_samples_split": [10, 20],
}

n_pos_train1 = int(y_train1.sum())
cv_folds = min(5, n_pos_train1)

if cv_folds >= 2:
    grid1 = GridSearchCV(
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        PARAM_GRID_P1, scoring="roc_auc", cv=cv_folds, n_jobs=-1, refit=True,
    )
    grid1.fit(X_train1, y_train1)
    phase1_model = grid1.best_estimator_
    print(f"[phase1 tuning] Best params: {grid1.best_params_} | Best CV AUC: {grid1.best_score_:.4f}")
    _ = evaluate_classifier("Phase 1 tuned DecisionTree", phase1_model, X_test1, y_test1)
else:
    phase1_model = phase1_baseline
    print(f"[phase1 tuning] Skipped -- only {n_pos_train1} positive example(s) in the training "
          f"split, not enough to cross-validate. Check SAMPLE_DIRS picked up all 20 samples.")

feature_importance1 = (
    pd.Series(phase1_model.feature_importances_, index=PHASE1_FEATURE_COLS)
    .sort_values(ascending=False)
)
print("\n[phase1] Top 10 feature importances:")
print(feature_importance1.head(10))

## 7. Phase 2 — Train the weekly re-scoring model

Every discharge episode contributes multiple rows (one per week) to `phase2_df`, so a plain
random train/test split would leak the same episode's earlier/later weeks across both sides.
The split below is **grouped by `discharge_row_id`** — an episode's weekly rows all land on
the same side of the split — combined with the same temporal ordering used in Phase 1.

In [ ]:
def group_temporal_split(df: pd.DataFrame, group_col: str, order_col: str, ratio: float = 0.8):
    episode_order = (
        df[[group_col, order_col]].drop_duplicates(group_col).sort_values(order_col)[group_col]
    )
    split_idx = int(len(episode_order) * ratio)
    train_groups = set(episode_order.iloc[:split_idx])
    train_mask = df[group_col].isin(train_groups)
    return df[train_mask].copy(), df[~train_mask].copy()


train2_df, test2_df = group_temporal_split(phase2_df, "discharge_row_id", "t0", ratio=0.8)
X_train2, y_train2 = train2_df[PHASE2_FEATURE_COLS], train2_df[LABEL_COL]
X_test2, y_test2 = test2_df[PHASE2_FEATURE_COLS], test2_df[LABEL_COL]

print(f"[phase2 split] train={len(train2_df)} rows / {train2_df['discharge_row_id'].nunique()} episodes "
      f"({y_train2.mean():.2%} positive) | "
      f"test={len(test2_df)} rows / {test2_df['discharge_row_id'].nunique()} episodes "
      f"({y_test2.mean():.2%} positive)")

phase2_baseline = DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)
phase2_baseline.fit(X_train2, y_train2)
_ = evaluate_classifier("Phase 2 baseline weekly DecisionTree", phase2_baseline, X_test2, y_test2)

In [ ]:
PARAM_GRID_P2 = PARAM_GRID_P1  # same search space as Phase 1

n_pos_train2 = int(y_train2.sum())
cv_folds2 = min(5, n_pos_train2)

if cv_folds2 >= 2:
    grid2 = GridSearchCV(
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        PARAM_GRID_P2, scoring="roc_auc", cv=cv_folds2, n_jobs=-1, refit=True,
    )
    grid2.fit(X_train2, y_train2)
    phase2_model = grid2.best_estimator_
    print(f"[phase2 tuning] Best params: {grid2.best_params_} | Best CV AUC: {grid2.best_score_:.4f}")
    _ = evaluate_classifier("Phase 2 tuned weekly DecisionTree", phase2_model, X_test2, y_test2)
else:
    phase2_model = phase2_baseline
    print(f"[phase2 tuning] Skipped -- only {n_pos_train2} positive training row(s), not enough "
          f"to cross-validate. Check SAMPLE_DIRS picked up all 20 samples.")

feature_importance2 = (
    pd.Series(phase2_model.feature_importances_, index=PHASE2_FEATURE_COLS)
    .sort_values(ascending=False)
)
print("\n[phase2] Top 10 feature importances:")
print(feature_importance2.head(10))

## 8. Trend detection layer

The feature proposal is explicit that a single weekly score isn't actionable on its own —
what drives care-team action is the **trend** across a patient's own recent weeks (§5-6 of
`Post_Discharge_Readmission_Monitoring (1).docx`). This applies the thresholds worked out in
`NewData.md`'s trend-engine sketch to the Phase 2 model's own weekly predicted probabilities:

| Trend | Condition | Action tier |
| --- | --- | --- |
| Sharp increase | ΔS ≥ +0.15, or a risk-band jump | Escalate immediately (same urgency as an in-hospital high-risk alert) |
| Increasing | ΔS > +0.03 over 2 consecutive weeks, or 3-week slope > 0 | Auto-alert care coordinator, prioritize outreach |
| Decreasing | ΔS < -0.02 | Log positive trend, consider reducing check-in cadence |
| Stable | \|ΔS\| ≤ 0.02 | Continue routine monitoring |

In [ ]:
def classify_trend(delta: float, slope: float | None) -> str:
    if pd.isna(delta):
        return "insufficient_history"
    if delta >= 0.15:
        return "sharp_increase"
    if delta > 0.03 or (slope is not None and not pd.isna(slope) and slope > 0):
        return "increasing"
    if delta < -0.02:
        return "decreasing"
    return "stable"


scored_test2 = test2_df.copy()
scored_test2["predicted_risk"] = phase2_model.predict_proba(X_test2)[:, 1]
scored_test2 = scored_test2.sort_values(["discharge_row_id", "week_number"])

scored_test2["prior_week_risk"] = scored_test2.groupby("discharge_row_id")["predicted_risk"].shift(1)
scored_test2["delta_risk"] = scored_test2["predicted_risk"] - scored_test2["prior_week_risk"]
scored_test2["rolling_slope_3wk"] = (
    scored_test2.groupby("discharge_row_id")["predicted_risk"]
    .transform(lambda s: s.rolling(3).apply(lambda w: np.polyfit(range(len(w)), w, 1)[0], raw=True))
)
scored_test2["trend"] = scored_test2.apply(
    lambda r: classify_trend(r["delta_risk"], r["rolling_slope_3wk"]), axis=1
)

print("[trend] Trend distribution across test-set weekly snapshots:")
print(scored_test2["trend"].value_counts())

print("\n[trend] Example trajectory for one episode:")
example_episode = scored_test2["discharge_row_id"].iloc[0]
print(scored_test2[scored_test2["discharge_row_id"] == example_episode][
    ["week_number", "predicted_risk", "delta_risk", "rolling_slope_3wk", "trend"]
])

## 9. Summary & next steps

- **Phase 1** produces one row per admission with several dozen engineered features spanning
  demographics, chronic conditions, CCI, discharge complexity, and 365d/90d prior-utilization
  lookbacks — trained with a temporally-split, class-balanced Decision Tree.
- **Phase 2** explodes each Phase 1 episode into weekly snapshots, layering cumulative
  follow-up/ED/pharmacy-adherence signal on top of the same static baseline, trained with an
  episode-grouped split so no episode's weeks leak across train/test, plus a trend-detection
  pass that turns the weekly score series into the proposal's Increasing / Sharp Increase /
  Decreasing / Stable action tiers.
- **Data spans all 20 CMS DE-SynPUF samples (~62 GB), processed one at a time** (§5) so the
  full dataset is never resident in memory at once — only the engineered `phase1_df`/
  `phase2_df` rows accumulate across samples. If a sample folder fails to load, it's skipped
  with a warning rather than aborting the whole run.
- **No Carrier Claims in this mirror**: PCP follow-up, ED-visit, and ambulance features are
  zero-filled for every patient (§0). If a future Kaggle attach *does* include Carrier
  Claims files, the loader picks them up automatically — no code change needed.
- Not modeled yet (per the feasibility doc's gap list): patient-reported symptoms/check-ins
  and care-coordination activity — those need Preventra's own product data once it's live,
  not CMS claims.